# False Sharing in OpenMP

You fixed the race condition
Your atomic code is correct
But it's still slow... why? → **False Sharing**

---

## What is a Cache Line?

CPUs don't load individual bytes from RAM - they load chunks called **cache lines** (typically **64 bytes**).

When Thread 0 writes to `hist[0]`, the CPU loads the **entire 64-byte line** into Thread 0's cache.  
When Thread 1 writes to `hist[1]`, it **invalidates** that line on Thread 0's cache - even though they touched different variables!

This is **false sharing** - the sharing is false because the threads aren't actually sharing data, but the CPU thinks they are.

## Setup

In [1]:
#include <omp.h>
#include <stdio.h>
#include <stdlib.h>
#include <string.h>

#define THREADS     4
#define ITERATIONS  100000000   // 100 million
#define CACHE_LINE  64          // bytes per cache line

printf("Threads:    %d\n", THREADS);
printf("Iterations: %d\n", ITERATIONS);
printf("Cache line: %d bytes\n", CACHE_LINE);
printf("sizeof(int) = %zu bytes\n", sizeof(int));
printf("4 ints fit in %zu bytes (one cache line = %d bytes)\n",
       4 * sizeof(int), CACHE_LINE);

Threads:    4
Iterations: 100000000
Cache line: 64 bytes
sizeof(int) = 4 bytes
4 ints fit in 16 bytes (one cache line = 64 bytes)


---
## Experiment 1 - False Sharing (Bad)

Each thread writes to its own `counter[thread_id]` - **different variables**, so no race condition.  
But all 4 counters fit in **one cache line** → every write invalidates the line for all other threads.

In [2]:
void false_sharing_demo() {
    // All 4 counters sit next to each other in memory = one cache line
    int counters[THREADS];
    memset(counters, 0, sizeof(counters));

    double t0 = omp_get_wtime();

    #pragma omp parallel num_threads(THREADS)
    {
        int tid = omp_get_thread_num();
        for (int i = 0; i < ITERATIONS / THREADS; i++) {
            counters[tid]++;   // No race condition, but FALSE SHARING!
        }
    }

    double elapsed = omp_get_wtime() - t0;

    long long total = 0;
    for (int t = 0; t < THREADS; t++) total += counters[t];
    printf("False sharing:  %.4f s  (total = %lld)\n", elapsed, total);
}

false_sharing_demo();

False sharing:  0.2637 s  (total = 100000000)


---
## Experiment 2 - No False Sharing (Padded)

We pad each counter so it occupies a **full 64-byte cache line** by itself.  
Now each thread's counter is on its own cache line → writes don't interfere.

```
Cache line 1 (64 bytes)         Cache line 2 (64 bytes)
┌────────┬──────────────────┐   ┌────────┬──────────────────┐
│counter0│    padding(60B)  │   │counter1│    padding(60B)  │
└────────┴──────────────────┘   └────────┴──────────────────┘
     Thread 0 only                   Thread 1 only
```

In [3]:
void no_false_sharing_demo() {
    // Pad each counter to 64 bytes so each lives on its own cache line
    struct {
        int value;
        char padding[60];   // 4 + 60 = 64 bytes = one full cache line
    } counters[THREADS];

    for (int t = 0; t < THREADS; t++) counters[t].value = 0;

    double t0 = omp_get_wtime();

    #pragma omp parallel num_threads(THREADS)
    {
        int tid = omp_get_thread_num();
        for (int i = 0; i < ITERATIONS / THREADS; i++) {
            counters[tid].value++;   // Each thread has its own cache line
        }
    }

    double elapsed = omp_get_wtime() - t0;

    long long total = 0;
    for (int t = 0; t < THREADS; t++) total += counters[t].value;
    printf("Padded (no FS): %.4f s  (total = %lld)\n", elapsed, total);
}

no_false_sharing_demo();

Padded (no FS): 0.0347 s  (total = 100000000)


---
## Experiment 3 - Private Variables (Best)

Even better than padding: use **thread-private local variables** on the stack.  
Stack variables are guaranteed to be in the thread's own memory - no sharing at all.

In [4]:
void private_vars_demo() {
    long long total = 0;

    double t0 = omp_get_wtime();

    #pragma omp parallel num_threads(THREADS) reduction(+:total)
    {
        long long local = 0;   // lives on THIS thread's stack - completely private
        for (int i = 0; i < ITERATIONS / THREADS; i++) {
            local++;           // no sharing of any kind
        }
        total += local;        // reduction merges at the end
    }

    double elapsed = omp_get_wtime() - t0;
    printf("Private vars:   %.4f s  (total = %lld)\n", elapsed, total);
}

private_vars_demo();

Private vars:   0.0433 s  (total = 100000000)


---
## Experiment 4 - All Three Side by Side with Speedup

In [5]:
void run_all() {
    double t0, t_fs, t_pad, t_priv;
    long long total;

    //False Sharing
    int counters_bad[THREADS];
    memset(counters_bad, 0, sizeof(counters_bad));
    t0 = omp_get_wtime();
    #pragma omp parallel num_threads(THREADS)
    {
        int tid = omp_get_thread_num();
        for (int i = 0; i < ITERATIONS / THREADS; i++)
            counters_bad[tid]++;
    }
    t_fs = omp_get_wtime() - t0;

    //Padded
    struct { int value; char pad[60]; } counters_good[THREADS];
    for (int t = 0; t < THREADS; t++) counters_good[t].value = 0;
    t0 = omp_get_wtime();
    #pragma omp parallel num_threads(THREADS)
    {
        int tid = omp_get_thread_num();
        for (int i = 0; i < ITERATIONS / THREADS; i++)
            counters_good[tid].value++;
    }
    t_pad = omp_get_wtime() - t0;

    //Private
    total = 0;
    t0 = omp_get_wtime();
    #pragma omp parallel num_threads(THREADS) reduction(+:total)
    {
        long long local = 0;
        for (int i = 0; i < ITERATIONS / THREADS; i++)
            local++;
        total += local;
    }
    t_priv = omp_get_wtime() - t0;

    //Results
    printf("  Method            Time      Speedup vs FS\n");
    printf("--------------------------------------------\n");
    printf("  False sharing:   %.4f s    1.0x\n",   t_fs);
    printf("  Padded:          %.4f s    %.1fx\n",  t_pad, t_fs / t_pad);
    printf("  Private vars:    %.4f s    %.1fx\n",  t_priv, t_fs / t_priv);
}

run_all();

  Method            Time      Speedup vs FS
--------------------------------------------
  False sharing:   0.2729 s    1.0x
  Padded:          0.0315 s    8.7x
  Private vars:    0.0425 s    6.4x
